# Time Evolution with ApproxTimeEvolution

Evolve a state under a ZZ + transverse field Hamiltonian using `qml.ApproxTimeEvolution` (Trotter-Suzuki decomposition) and compare against exact matrix-exponential evolution.

In [ ]:
import numpy as np
import pennylane as qml

## Hamiltonian and exact unitary

In [ ]:
N_QUBITS = 2
dev = qml.device("default.qubit", wires=N_QUBITS)
J, H_FIELD = 1.0, 0.5
HAMILTONIAN = qml.Hamiltonian(
    [J, H_FIELD, H_FIELD],
    [qml.Z(0) @ qml.Z(1), qml.X(0), qml.X(1)],
)

def exact_unitary(t):
    H_mat = qml.matrix(HAMILTONIAN, wire_order=[0, 1])
    eigenvalues, eigenvectors = np.linalg.eigh(H_mat)
    exp_diag = np.exp(-1j * eigenvalues * t)
    return eigenvectors @ np.diag(exp_diag) @ eigenvectors.conj().T

@qml.qnode(dev)
def trotter_circuit(t, n_steps, init_state="01"):
    if init_state == "11":
        qml.X(wires=0)
        qml.X(wires=1)
    elif init_state == "01":
        qml.X(wires=1)
    qml.ApproxTimeEvolution(HAMILTONIAN, t, n_steps)
    return qml.probs(wires=range(N_QUBITS))

## Compare exact vs Trotter evolution

In [ ]:
times = [0.5, 1.0, 2.0, 5.0]
n_trotter = 20

for t in times:
    trotter_probs = trotter_circuit(t, n_trotter)
    U = exact_unitary(t)
    psi0 = np.zeros(4, dtype=complex)
    psi0[1] = 1.0
    psi_t = U @ psi0
    exact = np.abs(psi_t) ** 2
    print(f"t={t:.1f}  exact: {[f'{p:.4f}' for p in exact]}  Trotter: {[f'{p:.4f}' for p in trotter_probs]}")

## Trotter convergence

In [ ]:
t_fixed = 2.0
U_exact = exact_unitary(t_fixed)
psi0 = np.zeros(4, dtype=complex)
psi0[1] = 1.0
psi_exact = U_exact @ psi0

for n_steps in [1, 2, 5, 10, 20, 50]:
    U_approx = qml.matrix(qml.ApproxTimeEvolution(HAMILTONIAN, t_fixed, n_steps), wire_order=[0, 1])
    psi_approx = U_approx @ psi0
    fidelity = float(np.abs(np.dot(psi_exact.conj(), psi_approx)) ** 2)
    print(f"  steps={n_steps:>3d}  fidelity={fidelity:.6f}")